In [1]:
import spatialdata as sd
import spatialdata_plot
import matplotlib.pyplot as plt
import numpy as np
import os
from skimage import io

/home/stefano/miniconda3/envs/analysis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = "/home/stefano/Documents/Spatial-Transcriptomic/data/blocco1_sham"
sdata = sd.read_zarr(data_path)
sdata

/tmp/ipykernel_17359/1828072172.py:2: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(data_path)
/home/stefano/miniconda3/envs/analysis/lib/python3.11/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


SpatialData object, with associated Zarr store: /home/stefano/Documents/Spatial-Transcriptomic/data/blocco1_sham
├── Images
│     ├── 'blocco1_hires_image': DataArray[cyx] (3, 1849, 4270)
│     ├── 'blocco1_lowres_image': DataArray[cyx] (3, 185, 427)
│     └── 'fluo_image': DataTree[cyx] (3, 7000, 16166), (3, 3500, 8083), (3, 1750, 4041), (3, 875, 2020)
├── Shapes
│     ├── 'GFP_poly': GeoDataFrame shape: (2, 5) (2D shapes)
│     ├── 'blocco1_square_008um': GeoDataFrame shape: (171116, 2) (2D shapes)
│     ├── 'blocco1_square_016um': GeoDataFrame shape: (44363, 1) (2D shapes)
│     ├── 'intissue_008um': GeoDataFrame shape: (95232, 3) (2D shapes)
│     └── 'intissue_poly': GeoDataFrame shape: (1, 5) (2D shapes)
└── Tables
      ├── 'filtered': AnnData (95232, 32285)
      ├── 'final_table': AnnData (95514, 32285)
      ├── 'square_008um': AnnData (95514, 32285)
      └── 'square_016um': AnnData (44363, 32285)
with coordinate systems:
    ▸ 'blocco1', with elements:
        blocco1_hires

In [99]:
from skimage import io as skio

In [98]:
mask= skio.imread("/home/stefano/Downloads/blocco1_sham_masks.tif")

In [ ]:
bin_shapes = sdata.shapes['intissue_008um']
#Estrai le coordinate pixel REALI dei bin sfruttando i centroidi della geometria
# Questo estrae i punti fisici (X, Y) nello spazio reale dell'immagine
centroids = bin_shapes.geometry.centroid
x_pixel = centroids.x.values.astype(int)
y_pixel = centroids.y.values.astype(int)

In [171]:
x_pixel

array([8088, 8063, 8038, ..., 5768, 5743, 5718], shape=(95232,))

In [107]:
#tabella delle annotazioni con info trascrizionali
adata_bins = sdata.tables['filtered']
print(adata_bins)

# Verifica di sicurezza: il numero di geometrie deve coincidere con le righe dell'AnnData
assert len(bin_shapes) == adata_bins.n_obs, "Discrepanza tra il numero di shapes e i bin"

AnnData object with n_obs × n_vars = 95232 × 32285
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'sample_id', 'in_treatment', 'GFP_value'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'


In [156]:
fiber_assignment = mask[(y_pixel - np.min(y_pixel)),(x_pixel- np.min(x_pixel))]
fiber_assignment

array([0, 0, 0, ..., 0, 0, 0], shape=(95232,), dtype=uint16)

In [161]:
# Salva l'informazione nell'AnnData
adata_bins.obs['assigned_fiber'] = fiber_assignment

# 6. Isola solo i bin associati a una fibra reale (ID > 0)
adata_inside_fibers = adata_bins[adata_bins.obs['assigned_fiber'] > 0].copy()

print(f"\nBin totali analizzati: {len(adata_bins)}")
print(f"Bin mappati dentro le fibre muscolari: {len(adata_inside_fibers)}")


Bin totali analizzati: 95232
Bin mappati dentro le fibre muscolari: 57061


In [175]:
adata_inside_fibers

AnnData object with n_obs × n_vars = 57061 × 32285
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'sample_id', 'in_treatment', 'GFP_value', 'assigned_fiber'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'